# 02 — Model Training & Comparison
**Solar Power 24-Hour Forecast | System 2107, Arbuckle CA**

**Prerequisites:** Run `01_data_pipeline.py` first to generate the `data/` CSV files.

| Split | Period |
|---|---|
| Train | 2022-03-23 → 2023-11-09 |
| Validation | 2024-01-01 → 2024-05-31 |
| Test | 2024-06-01 → 2024-10-31 |


In [ ]:
import pandas as pd
import numpy as np
import warnings, copy
warnings.filterwarnings("ignore")

import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler, LabelEncoder, QuantileTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec

USE_MLP = True   # set False to skip MLP for faster run
print("Imports OK")


## 1. Load Pre-Built Feature CSVs
Generated by `01_data_pipeline.py`

In [ ]:
# ── Load train / val / test feature sets from CSV ─────────────────────────────
df_train_feat = pd.read_csv("data/df_train_features.csv", index_col=0, parse_dates=True)
df_val_feat   = pd.read_csv("data/df_val_features.csv",   index_col=0, parse_dates=True)
df_test_feat  = pd.read_csv("data/df_test_features.csv",  index_col=0, parse_dates=True)

with open("data/feature_list.txt") as f:
    FEATURES = [line.strip() for line in f if line.strip()]

TARGET             = "actual_power_kw"
SYSTEM_CAPACITY_KW = 707.0

# Pre-test combined for final retraining
df_pretrain_feat = pd.concat([df_train_feat, df_val_feat]).sort_index()

X_train = df_train_feat[FEATURES];    y_train = df_train_feat[TARGET]
X_val   = df_val_feat[FEATURES];      y_val   = df_val_feat[TARGET]
X_test  = df_test_feat[FEATURES];     y_test  = df_test_feat[TARGET]
X_pre   = df_pretrain_feat[FEATURES]; y_pre   = df_pretrain_feat[TARGET]

print(f"Train    : {df_train_feat.index.min().date()} → {df_train_feat.index.max().date()}  {X_train.shape}")
print(f"Val      : {df_val_feat.index.min().date()}   → {df_val_feat.index.max().date()}    {X_val.shape}")
print(f"Test     : {df_test_feat.index.min().date()}  → {df_test_feat.index.max().date()}   {X_test.shape}")
print(f"Features : {len(FEATURES)}")
print(f"\nSanity checks:")
print(f"  y_test max     : {y_test.max():.1f} kW")
print(f"  lag_96 max     : {X_test['lag_96'].max():.1f} kW")
print(f"  NaNs in X_test : {X_test.isna().sum().sum()}")


In [ ]:
# Speed optimisations applied throughout:
#   - LightGBM/XGBoost: early stopping caps n_estimators; lower num_leaves for CV
#   - RandomForest: n_estimators reduced to 300 (enough for tabular solar data)
#   - MLP: smaller architecture (256,128,64), tighter early stopping patience
#   - All tree models use n_jobs=-1 (parallel)

# ── LightGBM ──────────────────────────────────────────────────────────────────
lgb_params = dict(
    num_leaves        = 127,
    learning_rate     = 0.1,       # ↑ faster convergence
    n_estimators      = 500,       # ↓ early stopping fires well before this,
    subsample         = 0.8,
    subsample_freq    = 1,
    colsample_bytree  = 0.8,
    min_child_samples = 30,        # ↑ reduces overfit on sparse high-power regions
    min_split_gain    = 0.01,      # require minimum gain to split — reduces overfit
    reg_alpha         = 0.1,       # ↑ L1 regularisation
    reg_lambda        = 1.0,       # ↑ L2 regularisation
    max_bin           = 255,
    random_state      = 42,
    verbose           = -1,
    n_jobs            = -1,
)

# ── XGBoost ───────────────────────────────────────────────────────────────────
xgb_params = dict(
    max_depth         = 6,
    learning_rate     = 0.1,       # ↑ faster convergence → fewer trees
    n_estimators      = 500,       # ↓ hard cap; early stopping fires ~100-200
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    colsample_bylevel = 0.8,
    min_child_weight  = 30,
    reg_alpha         = 0.1,
    reg_lambda        = 1.0,
    gamma             = 0.05,
    random_state      = 42,
    verbosity         = 0,
    n_jobs            = -1,
    tree_method       = "hist",
    early_stopping_rounds = 30,    # stop if no improvement for 30 rounds
)

# ── Random Forest ─────────────────────────────────────────────────────────────
rf_params = dict(
    n_estimators     = 200,        # ↓ 300→200: faster, accuracy ~same
    max_depth        = 15,
    min_samples_leaf = 10,
    max_features     = 0.5,
    random_state     = 42,
    n_jobs           = -1,
)

# ── Ridge ─────────────────────────────────────────────────────────────────────
ridge_params = dict(alpha=0.1)    # near-instant, no changes needed

# ── MLP ───────────────────────────────────────────────────────────────────────
mlp_params = dict(
    hidden_layer_sizes  = (256, 128, 64),  # smaller than 512,256,128,64: ~3× faster
    activation          = "relu",
    solver              = "adam",
    alpha               = 1e-4,
    batch_size          = 256,             # explicit batch size: faster per epoch
    learning_rate_init  = 3e-3,            # higher LR: fewer epochs to converge
    max_iter            = 150,             # ↓ 200→150 epochs max
    early_stopping      = True,
    validation_fraction = 0.1,
    n_iter_no_change    = 15,              # tighter patience: stops sooner
    random_state        = 42,
)

# ── Build registry ─────────────────────────────────────────────────────────────
# ── Linear Regression ────────────────────────────────────────────────────────
# Ordinary Least Squares — no regularisation. Uses same StandardScaler pipeline
# as Ridge. Included to show whether regularisation (Ridge) is beneficial over
# plain OLS for this solar forecasting task.
from sklearn.linear_model import LinearRegression

MODEL_REGISTRY_TEMPLATES = {
    "LightGBM":     lgb.LGBMRegressor(**lgb_params),
    "XGBoost":      xgb.XGBRegressor(**xgb_params),
    "RandomForest": RandomForestRegressor(**rf_params),
    "Ridge":        Pipeline([("scaler", StandardScaler()),
                               ("model", Ridge(**ridge_params))]),
    "LinearReg":    Pipeline([("scaler", StandardScaler()),
                               ("model", LinearRegression())]),
}

if USE_MLP:
    MODEL_REGISTRY_TEMPLATES["MLP"] = Pipeline([
        ("scaler", QuantileTransformer(output_distribution="normal",
                                       n_quantiles=min(1000, len(X_train)),
                                       random_state=42)),
        ("model",  MLPRegressor(**mlp_params)),
    ])
    print("MLP included.")
else:
    print("MLP skipped (USE_MLP = False).")

print(f"\nModel registry ({len(MODEL_REGISTRY_TEMPLATES)} models):")
for name in MODEL_REGISTRY_TEMPLATES:
    print(f"  {name}")

# ── Estimated runtimes (approximate, single-core equivalent) ──────────────────
print("\nEstimated final-fit times:")
print("  LightGBM     ~30s  (early stopping ~200-400 trees)")
print("  XGBoost      ~45s  (early stopping ~200-400 trees)")
print("  RandomForest ~60s  (300 trees, depth=20, parallel)")
print("  Ridge        <1s")
print("  MLP          ~90s  (256,128,64 adam, early stop)")
print("  Rolling CV   ~5min (52 folds × LightGBM+XGBoost+Ridge only)")


## 5. Rolling-Window CV & Final Training

**Protocol:**
1. **Rolling CV** — 365-day window slides daily through the train pool (2022-03-23 → 2023-11-09). Each fold scores on the next day. Folds with no daytime activity are skipped.
2. **Final refit** — each model retrained on all pre-2024 data (train + val) with 2024-Jan–May as early-stopping reference.
3. **Test evaluation** — scored on the held-out 2024-Jun–Oct period.


In [ ]:
SYSTEM_CAPACITY_KW = 707.0   # installed AC capacity — hard physical upper bound

def nighttime_zero(preds, df_feat):
    """Zero predictions when shortwave < 10 W/m² and clip to [0, capacity]."""
    night = df_feat["shortwave_radiation"].values < 10
    preds = np.where(night, 0, preds)
    return np.clip(preds, 0, SYSTEM_CAPACITY_KW)  # physical cap at system capacity

def rmse_pct(y_true, y_pred, mask=None):
    """RMSE% = RMSE(predicted - actual) / mean_daytime_actual × 100

    Formula: sqrt( mean( (predicted_i - actual_i)² ) ) / mean(actual_daytime) × 100

    - Computed over daytime rows only (actual > 0).
    - Optional boolean mask to exclude specific rows (e.g. anomaly days).
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    dt = y_true > 0
    if mask is not None:
        dt = dt & np.asarray(mask, dtype=bool)
    if dt.sum() == 0:
        return np.nan
    rmse        = np.sqrt(np.mean((y_pred[dt] - y_true[dt]) ** 2))
    mean_actual = y_true[dt].mean()
    return (rmse / mean_actual) * 100


def rmse_absolute(y_true, y_pred, nonzero_only=False):
    """Absolute RMSE in kW.

    Parameters
    ----------
    y_true       : array of actual power values (kW)
    y_pred       : array of predicted power values (kW)
    nonzero_only : if True, compute only on rows where actual > 0
                   (daytime rows); if False, include all rows (night zeros too)

    Returns
    -------
    RMSE in kW
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if nonzero_only:
        mask = y_true > 0
        if mask.sum() == 0:
            return np.nan
        return np.sqrt(np.mean((y_pred[mask] - y_true[mask]) ** 2))
    return np.sqrt(np.mean((y_pred - y_true) ** 2))


def flag_anomaly_days(y_true, index, threshold_pct=30):
    """Flag days where daily production is >threshold% below the rolling 14-day median.
    Returns a boolean mask: True = normal day, False = anomaly/cloudy day.
    """
    s = pd.Series(y_true, index=index)
    daily_sum  = s.resample("D").sum()
    roll_med   = daily_sum.rolling(14, min_periods=3, center=True).median()
    pct_of_med = daily_sum / roll_med.clip(lower=1) * 100
    anomaly_dates = set(pct_of_med[pct_of_med < (100 - threshold_pct)].index.date)
    normal_mask = pd.Series(
        [d.date() not in anomaly_dates for d in index],
        index=index
    )
    return normal_mask.values, anomaly_dates

def clone_model(name):
    return copy.deepcopy(MODEL_REGISTRY_TEMPLATES[name])

def fit_model(name, m, X_tr, y_tr, X_va, y_va):
    """Fit with early stopping where the model supports eval_set."""
    if isinstance(m, lgb.LGBMRegressor):
        m.fit(X_tr, y_tr,
              eval_set=[(X_va, y_va)],
              callbacks=[lgb.early_stopping(50, verbose=False),
                         lgb.log_evaluation(period=-1)])
    elif isinstance(m, xgb.XGBRegressor):
        # early_stopping_rounds set in params; verbose=False suppresses output
        m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    else:
        m.fit(X_tr, y_tr)
    return m

# ── Rolling-window CV ─────────────────────────────────────────────────────────
# Weekly steps (~52 folds) × fast models only → keeps runtime ~5 min
WINDOW_DAYS = 365
STEP_DAYS   = 7
SCORE_DAYS  = 7
# CV uses LightGBM only — fastest model, representative of tree performance
# XGBoost/RF are similar enough; MLP too slow for 52-fold CV
CV_MODELS   = ["LightGBM", "Ridge"]   # ~2min total

pool_start  = df_train_feat.index.min()
pool_end    = df_train_feat.index.max()

fold_starts = pd.date_range(
    start = pool_start,
    end   = pool_end - pd.Timedelta(days=WINDOW_DAYS + SCORE_DAYS),
    freq  = f"{STEP_DAYS}D",
)

print(f"Rolling CV  window={WINDOW_DAYS}d  step={STEP_DAYS}d  score={SCORE_DAYS}d")
print(f"Pool        {pool_start.date()} → {pool_end.date()}")
print(f"Folds       {len(fold_starts)}  |  CV models: {CV_MODELS}")
print()

cv_records = {name: [] for name in CV_MODELS}

for fold_i, fs in enumerate(fold_starts):
    fe        = fs + pd.Timedelta(days=WINDOW_DAYS)
    score_end = fe + pd.Timedelta(days=SCORE_DAYS)

    fold_tr = df_train_feat[(df_train_feat.index >= fs) & (df_train_feat.index < fe)]
    fold_sc = df_train_feat[(df_train_feat.index >= fe) & (df_train_feat.index < score_end)]

    if len(fold_tr) < 100 or len(fold_sc) < 10:
        continue
    if (fold_sc[TARGET] > 10).sum() == 0:
        continue

    # Subsample CV training to daytime rows — halves rows, same signal
    dt_tr = fold_tr[TARGET] > 0
    X_tr_f = fold_tr.loc[dt_tr, FEATURES]
    y_tr_f = fold_tr.loc[dt_tr, TARGET]
    X_sc_f, y_sc_f = fold_sc[FEATURES], fold_sc[TARGET]

    for name in CV_MODELS:
        m = clone_model(name)
        fit_model(name, m, X_tr_f, y_tr_f, X_sc_f, y_sc_f)
        p = nighttime_zero(m.predict(X_sc_f), fold_sc)
        cv_records[name].append({
            "fold":       fold_i,
            "score_date": fe.date(),
            "rmse_pct":   rmse_pct(y_sc_f.values, p),
        })

    print(f"  Fold {fold_i+1:>2}/{len(fold_starts)} | {fs.date()} → {fe.date()}")

cv_dfs = {name: pd.DataFrame(recs).dropna(subset=["rmse_pct"])
          for name, recs in cv_records.items()}

print("\nRolling CV RMSE%:")
print(f"  {'Model':<15} {'Mean':>8} {'Std':>7} {'Min':>7} {'Max':>8}")
print("  " + "-"*50)
for name, df_ in cv_dfs.items():
    v = df_["rmse_pct"]
    print(f"  {name:<15} {v.mean():>7.2f}% {v.std():>6.2f}%"
          f" {v.min():>6.2f}% {v.max():>7.2f}%")

# ── Pre-flight check ──────────────────────────────────────────────────────────
print("\nPre-flight sanity check before final training:")
print(f"  y_test max            : {y_test.max():.1f} kW")
print(f"  y_test daytime mean   : {y_test[y_test>10].mean():.1f} kW")
print(f"  lag_96 test max       : {X_test['lag_96'].max():.1f}")
print(f"  lag_96 test NaNs      : {X_test['lag_96'].isna().sum()}")
print(f"  X_test NaNs total     : {X_test.isna().sum().sum()}")
print(f"  target == y_test      : {(df_test_feat[TARGET].values == y_test.values).all()}")

# ── Flag anomaly/cloudy days in test set ─────────────────────────────────────
clear_mask_test, anomaly_dates_test = flag_anomaly_days(
    y_test.values, df_test_feat.index, threshold_pct=30
)
print(f"Anomaly days in test set ({len(anomaly_dates_test)} days):")
for d in sorted(anomaly_dates_test):
    day_mask = df_test_feat.index.date == d
    daily_prod = y_test[day_mask].sum()
    print(f"  {d}  daily production={daily_prod:.0f} kWh")
print()

# ── Final refit on all pre-test data ─────────────────────────────────────────────
print("\nFinal retraining on all pre-test data (train + val)...")
results = {}
for name in MODEL_REGISTRY_TEMPLATES:
    m = clone_model(name)
    print(f"  {name}...", end=" ", flush=True)
    fit_model(name, m, X_pre, y_pre, X_val, y_val)

    preds_test = nighttime_zero(m.predict(X_test), df_test_feat)
    preds_val  = nighttime_zero(m.predict(X_val),  df_val_feat)
    dt_test    = y_test > 10

    results[name] = {
        "model":        m,
        "preds":        preds_test,
        "preds_val":    preds_val,
        "rmse_pct":     rmse_pct(y_test.values, preds_test),
        "rmse_pct_clear": rmse_pct(y_test.values, preds_test, mask=clear_mask_test),
        "val_rmse_pct": rmse_pct(y_val.values,  preds_val),
        "mae":          mean_absolute_error(y_test, preds_test),
        "r2":           r2_score(y_test[dt_test], preds_test[dt_test]),
        "cv_mean":      cv_dfs[name]["rmse_pct"].mean() if name in cv_dfs else np.nan,
        "cv_std":       cv_dfs[name]["rmse_pct"].std()  if name in cv_dfs else np.nan,
    }
    print(f"Val RMSE%={results[name]['val_rmse_pct']:.2f}%  "
          f"Test RMSE%={results[name]['rmse_pct']:.2f}%  "
          f"Clear-day RMSE%={results[name]['rmse_pct_clear']:.2f}%  "
          f"R²={results[name]['r2']:.4f}")

summary = pd.DataFrame({
    name: {
        "CV RMSE% (mean)":         r["cv_mean"],
        "CV RMSE% (std)":          r["cv_std"],
        "Val RMSE% (2024-H1)":     r["val_rmse_pct"],
        "Test RMSE% (all days)":   r["rmse_pct"],
        "Test RMSE% (clear days)": r.get("rmse_pct_clear", np.nan),
        "Test MAE (kW)":           r["mae"],
        "Test R² daytime":         r["r2"],
    }
    for name, r in results.items()
}).T.sort_values("Test RMSE% (all days)")

print("\n" + "="*75)
print("  MODEL COMPARISON | Rolling CV | Val=2024-H1 | Test=2024-H2")
print("="*75)
print(summary.round(3).to_string())
print("="*75)

# ─────────────────────────────────────────────────────────────────────────────
# ── Stacking ensemble — meta-learner on base model predictions ───────────────
# Train a Ridge on top of all base models' val-set predictions (out-of-fold).
# Using Ridge with positive=True so blend weights are non-negative — ensures
# the ensemble is a convex combination that can't extrapolate wildly.
# This captures complementary strengths: e.g. LightGBM handles morning ramp,
# RandomForest the midday plateau, MLP the tail-off shape.
print("\nBuilding stacking ensemble...")

from sklearn.linear_model import Ridge as _RidgeMeta
from sklearn.preprocessing import StandardScaler as _SSMeta

base_names = [n for n in results if results[n]["model"] is not None]

# Meta-features on val set (same period the meta-learner is trained on)
meta_val_X = np.column_stack([results[n]["preds_val"] for n in base_names])
meta_val_y = y_val.values

# Meta-features on test set
meta_test_X = np.column_stack([results[n]["preds"] for n in base_names])

# Scale so Ridge penalty is applied equally across all base models
_meta_sc = _SSMeta()
meta_val_Xs  = _meta_sc.fit_transform(meta_val_X)
meta_test_Xs = _meta_sc.transform(meta_test_X)

# positive=True: blend weights ≥ 0 → true convex combination
_meta = _RidgeMeta(alpha=1.0, positive=True)
_meta.fit(meta_val_Xs, meta_val_y)

print("  Blend weights:")
for n, w in zip(base_names, _meta.coef_):
    print(f"    {n:<15} {w:.4f}")

stack_preds_raw = _meta.predict(meta_test_Xs)
stack_preds     = nighttime_zero(
    stack_preds_raw,
    df_test_feat
)
stack_preds_val_raw = _meta.predict(meta_val_Xs)
stack_preds_val     = nighttime_zero(stack_preds_val_raw, df_val_feat)

dt_test = y_test > 10
# Store meta-learner separately — do NOT put in model slot.
# Downstream cells call r["model"].predict(X_window) with raw features,
# which would fail because _meta expects 5 meta-features not 57 raw features.
# Instead set model=None so those cells use pre-computed preds (like baselines).
_stacking_meta   = _meta          # keep reference for inspection
_stacking_meta_sc = _meta_sc      # keep scaler reference

results["Stacking"] = {
    "model":          None,        # None = use pre-computed preds, not raw predict
    "preds":          stack_preds,
    "preds_val":      stack_preds_val,
    "rmse_pct":       rmse_pct(y_test.values, stack_preds),
    "rmse_pct_clear": rmse_pct(y_test.values, stack_preds, mask=clear_mask_test),
    "val_rmse_pct":   rmse_pct(y_val.values, stack_preds_val),
    "mae":            mean_absolute_error(y_test, stack_preds),
    "r2":             r2_score(y_test[dt_test], stack_preds[dt_test]),
    "cv_mean":        np.nan,
    "cv_std":         np.nan,
}
print(f"  Stacking Test RMSE%       : {results['Stacking']['rmse_pct']:.2f}%")
print(f"  Stacking Clear-day RMSE%  : {results['Stacking']['rmse_pct_clear']:.2f}%")
print(f"  Stacking Val  RMSE%       : {results['Stacking']['val_rmse_pct']:.2f}%")

# Rebuild summary including Stacking
summary = pd.DataFrame({
    name: {
        "CV RMSE% (mean)":         r["cv_mean"],
        "CV RMSE% (std)":          r["cv_std"],
        "Val RMSE% (2024-H1)":     r["val_rmse_pct"],
        "Test RMSE% (all days)":   r["rmse_pct"],
        "Test RMSE% (clear days)": r.get("rmse_pct_clear", np.nan),
        "Test MAE (kW)":           r["mae"],
        "Test R² daytime":         r["r2"],
    }
    for name, r in results.items()
}).T.sort_values("Test RMSE% (all days)")

print("\n" + "="*75)
print("  MODEL COMPARISON incl. Stacking | Val=2024-H1 | Test=2024-H2")
print("="*75)
print(summary.round(3).to_string())
print("="*75)


## 5b. Baseline Models
Two standard solar forecasting baselines used to benchmark model skill:

| Baseline | Description |
|---|---|
| **Persistence** | Predicts today's power = same 15-min slot from yesterday (`lag_96`) |
| **Climatology** | Predicts each 15-min slot = mean of that slot across all training days |
| **Smart Persistence** | Scales yesterday's value by today's clear-sky irradiance ratio |

A model with RMSE% significantly below the baselines is genuinely adding value beyond naive reference forecasts.


In [ ]:
# ── Persistence — lag-96 (same 15-min slot from 24h ago) ────────────────────
persist_test = nighttime_zero(df_test_feat["lag_96"].fillna(0).values, df_test_feat)
persist_val  = nighttime_zero(df_val_feat["lag_96"].fillna(0).values,  df_val_feat)

# ── Climatology — mean by (month, hour, minute) from training set ─────────────
clim = (df_train_feat.assign(
            month  = lambda d: d.index.month,
            hour   = lambda d: d.index.hour,
            minute = lambda d: d.index.minute,
        )
        .groupby(["month","hour","minute"])[TARGET]
        .mean()
        .rename("clim_mean"))   # MultiIndex Series keyed by (month, hour, minute)

def apply_climatology(df_feat):
    keys   = list(zip(df_feat.index.month, df_feat.index.hour, df_feat.index.minute))
    preds  = np.array([clim.get(k, 0.0) for k in keys], dtype=float)
    preds  = np.nan_to_num(preds, nan=0.0)
    return nighttime_zero(preds, df_feat)

clim_test = apply_climatology(df_test_feat)
clim_val  = apply_climatology(df_val_feat)

# ── Smart Persistence — yesterday scaled by today/yesterday shortwave ratio ───
def smart_persistence(df_feat):
    sw_today = df_feat["shortwave_radiation"].values
    sw_yest  = df_feat["shortwave_radiation"].shift(96).bfill().values
    ratio    = np.where(sw_yest > 10, sw_today / sw_yest, 1.0)
    ratio    = np.clip(ratio, 0, 1.5)   # cap at 1.5× — prevents spikes on post-cloud days
    preds    = df_feat["lag_96"].fillna(0).values * ratio
    return nighttime_zero(preds, df_feat)

sp_test = smart_persistence(df_test_feat)
sp_val  = smart_persistence(df_val_feat)

# ── Package baselines ─────────────────────────────────────────────────────────
dt_test = y_test > 10
BASELINE_NAMES = ["Persistence", "Climatology", "SmartPersistence"]

for bname, preds_t, preds_v in [
    ("Persistence",      persist_test, persist_val),
    ("Climatology",      clim_test,    clim_val),
    ("SmartPersistence", sp_test,      sp_val),
]:
    results[bname] = {
        "model":        None,
        "preds":        preds_t,
        "preds_val":    preds_v,
        "rmse_pct":       rmse_pct(y_test.values, preds_t),
        "rmse_pct_clear": rmse_pct(y_test.values, preds_t, mask=clear_mask_test),
        "val_rmse_pct":   rmse_pct(y_val.values,  preds_v),
        "rmse_kw":        rmse_absolute(y_test.values, preds_t, nonzero_only=True),
        "rmse_kw_all":    rmse_absolute(y_test.values, preds_t, nonzero_only=False),
        "mae":          mean_absolute_error(y_test, preds_t),
        "r2":           r2_score(y_test[dt_test], preds_t[dt_test]),
        "cv_mean":      np.nan,
        "cv_std":       np.nan,
    }
    print(f"{bname:<22}  Val RMSE%={results[bname]['val_rmse_pct']:.2f}%  "
          f"Test RMSE%={results[bname]['rmse_pct']:.2f}%")

# ── Rebuild summary ───────────────────────────────────────────────────────────
summary = pd.DataFrame({
    name: {
        "CV RMSE% (mean)":         r["cv_mean"],
        "CV RMSE% (std)":          r["cv_std"],
        "Val RMSE% (2024-H1)":     r["val_rmse_pct"],
        "Test RMSE% (all days)":   r["rmse_pct"],
        "Test RMSE% (clear days)": r.get("rmse_pct_clear", np.nan),
        "Test RMSE kW (daytime)":  r.get("rmse_kw", np.nan),
        "Test RMSE kW (all hrs)":  r.get("rmse_kw_all", np.nan),
        "Test MAE (kW)":           r["mae"],
        "Test R² daytime":         r["r2"],
    }
    for name, r in results.items()
}).T.sort_values("Test RMSE% (all days)")

print("\n" + "="*75)
print("  FULL COMPARISON incl. Baselines")
print("="*75)
print(summary.round(3).to_string())
print("="*75)

best_baseline_rmse = min(results[b]["rmse_pct"] for b in BASELINE_NAMES)
best_baseline_name = min(BASELINE_NAMES, key=lambda b: results[b]["rmse_pct"])
print(f"\nFSS vs {best_baseline_name} ({best_baseline_rmse:.2f}%):")
for name in [n for n in results if results[n]["model"] is not None and n not in BASELINE_NAMES]:
    fss = 1 - results[name]["rmse_pct"] / best_baseline_rmse
    print(f"  {name:<15}  FSS = {fss:+.3f}")


## 6. Rolling CV RMSE% Over Time

In [ ]:
import matplotlib as mpl
mpl.rcParams.update({
    "font.size":         18,
    "axes.titlesize":    22,
    "axes.labelsize":    20,
    "xtick.labelsize":   17,
    "ytick.labelsize":   17,
    "legend.fontsize":   16,
    "axes.linewidth":    1.4,
    "axes.spines.top":   False,
    "axes.spines.right": False,
})

COLORS = {
    "LightGBM":          "#2196F3",
    "XGBoost":           "#FF5722",
    "RandomForest":      "#4CAF50",
    "Ridge":             "#9C27B0",
    "LinearReg":         "#00BCD4",  # cyan
    "MLP":               "#FF9800",   # only shown if USE_MLP=True
    "Persistence":       "#607D8B",
    "Climatology":       "#795548",
    "SmartPersistence":  "#009688",
    "Stacking":          "#F44336",  # red — the ensemble
}
LINESTYLES = {
    "LightGBM":         (0, ()),
    "XGBoost":          (0, (6, 2)),
    "RandomForest":     (0, (2, 2)),
    "Ridge":            (0, (4, 2, 1, 2)),
    "LinearReg":        (0, (2, 1)),
    "MLP":              (0, (1, 1)),
    "Persistence":      (0, (3, 1, 1, 1, 1, 1)),
    "Climatology":      (0, (5, 5)),
    "SmartPersistence": (0, (3, 1)),
    "Stacking":         (0,()),      # solid red — ensemble
}
LINEWIDTHS = {
    "LightGBM": 2.5, "XGBoost": 2.2, "RandomForest": 2.2, "Ridge": 2.2, "LinearReg": 2.0,
    "MLP": 2.2, "Persistence": 1.8, "Climatology": 1.8, "SmartPersistence": 1.8,
    "Stacking":          3.0,
}
# Only show models that were actually trained
MODEL_NAMES = [n for n in COLORS if n in results]


fig, ax = plt.subplots(figsize=(22, 8))
for name, df_ in cv_dfs.items():  # only fast models
    dates     = pd.to_datetime(df_["score_date"])
    roll_mean = df_["rmse_pct"].rolling(7, min_periods=1).mean()
    roll_std  = df_["rmse_pct"].rolling(7, min_periods=1).std().fillna(0)
    ax.plot(dates, roll_mean, color=COLORS[name], linewidth=LINEWIDTHS[name],
            linestyle=LINESTYLES[name], alpha=0.9, label=name)
    ax.fill_between(dates, roll_mean - roll_std, roll_mean + roll_std,
                    color=COLORS[name], alpha=0.08)

ax.set_title("Rolling CV RMSE% — Daily Folds (Training Pool)", pad=16)
ax.set_ylabel("RMSE%")
ax.set_xlabel("Score date")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1,4,7,10]))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")
ax.legend(ncol=3, framealpha=0.4)
plt.tight_layout()
plt.savefig("solar_rolling_cv.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → solar_rolling_cv.png")


## 7. 24-Hour Forecast Walk-Through
A 7-day window from the **test set (Jun–Oct 2024)**.

In [ ]:
forecast_start = pd.Timestamp("2024-07-15")
forecast_end   = forecast_start + pd.Timedelta(days=7)

window   = df_test_feat[forecast_start:forecast_end]
X_window = window[FEATURES]
y_window = window[TARGET]

MODEL_NAMES = list(results.keys())

window_preds = {}
for name, r in results.items():
    if r["model"] is not None:
        # Standard ML model — predict from raw features
        raw = r["model"].predict(X_window)
        window_preds[name] = nighttime_zero(raw, window)
    elif name == "Stacking":
        # Meta-learner needs meta-features (base model predictions), not raw features
        meta_window_X = np.column_stack([
            window_preds[n] for n in base_names if n in window_preds
        ])
        if meta_window_X.shape[1] == len(base_names):
            meta_window_Xs = _stacking_meta_sc.transform(meta_window_X)
            raw_stack = _stacking_meta.predict(meta_window_Xs)
            window_preds[name] = nighttime_zero(raw_stack, window)
        else:
            # Fallback: slice from pre-computed test preds if base models not ready
            mask = (df_test_feat.index >= forecast_start) & (df_test_feat.index <= forecast_end)
            window_preds[name] = r["preds"][mask]
    else:
        # Baselines — slice from pre-computed full test predictions
        mask = (df_test_feat.index >= forecast_start) & (df_test_feat.index <= forecast_end)
        window_preds[name] = r["preds"][mask]

print(f"7-day window : {forecast_start.date()} → {forecast_end.date()}  ({len(window)} rows)")

# ── 30-day forecast using best ML model ───────────────────────────────────────
best_ml = min(
    [n for n in results if results[n]["model"] is not None],
    key=lambda n: results[n]["rmse_pct"]
)
print(f"Best ML model for 30-day forecast: {best_ml}")

forecast_30_start = pd.Timestamp("2024-07-01")
forecast_30_end   = forecast_30_start + pd.Timedelta(days=30)
window_30   = df_test_feat[forecast_30_start:forecast_30_end]
X_window_30 = window_30[FEATURES]
y_window_30 = window_30[TARGET]

preds_30_raw = results[best_ml]["model"].predict(X_window_30)
preds_30     = nighttime_zero(preds_30_raw, window_30)
rmse_30      = rmse_pct(y_window_30.values, preds_30)
print(f"30-day forecast RMSE%: {rmse_30:.2f}%")


In [ ]:
daily_rmse_pct = {}
dates = pd.date_range(forecast_start, forecast_end - pd.Timedelta(days=1), freq="D")

for name in results:
    day_errors = []
    for d in dates:
        mask = (window.index >= d) & (window.index < d + pd.Timedelta(days=1))
        y_d = y_window[mask].values
        p_d = window_preds[name][mask]
        dt  = y_d > 10
        if dt.sum() == 0:
            day_errors.append(np.nan)
        else:
            r = np.sqrt(mean_squared_error(y_d[dt], p_d[dt]))
            day_errors.append((r / y_d[dt].mean()) * 100)
    daily_rmse_pct[name] = day_errors

daily_df = pd.DataFrame(daily_rmse_pct, index=[d.strftime("%b %d") for d in dates])
print("Per-day RMSE% — Forecast Window")
print(daily_df.round(2).to_string())


## 8. Diagnostic & Comparison Plots

In [ ]:
import matplotlib as mpl
mpl.rcParams.update({
    "font.size":         18,
    "axes.titlesize":    22,
    "axes.labelsize":    20,
    "xtick.labelsize":   17,
    "ytick.labelsize":   17,
    "legend.fontsize":   16,
    "axes.linewidth":    1.4,
    "axes.spines.top":   False,
    "axes.spines.right": False,
})

COLORS = {
    "LightGBM":          "#2196F3",
    "XGBoost":           "#FF5722",
    "RandomForest":      "#4CAF50",
    "Ridge":             "#9C27B0",
    "LinearReg":         "#00BCD4",  # cyan
    "MLP":               "#FF9800",   # only shown if USE_MLP=True
    "Persistence":       "#607D8B",
    "Climatology":       "#795548",
    "SmartPersistence":  "#009688",
    "Stacking":          "#F44336",  # red — the ensemble
}
LINESTYLES = {
    "LightGBM":         (0, ()),
    "XGBoost":          (0, (6, 2)),
    "RandomForest":     (0, (2, 2)),
    "Ridge":            (0, (4, 2, 1, 2)),
    "LinearReg":        (0, (2, 1)),
    "MLP":              (0, (1, 1)),
    "Persistence":      (0, (3, 1, 1, 1, 1, 1)),
    "Climatology":      (0, (5, 5)),
    "SmartPersistence": (0, (3, 1)),
    "Stacking":         (0,()),      # solid red — ensemble
}
LINEWIDTHS = {
    "LightGBM": 2.5, "XGBoost": 2.2, "RandomForest": 2.2, "Ridge": 2.2, "LinearReg": 2.0,
    "MLP": 2.2, "Persistence": 1.8, "Climatology": 1.8, "SmartPersistence": 1.8,
    "Stacking":          3.0,
}
# Only show models that were actually trained
MODEL_NAMES = [n for n in COLORS if n in results]


zoom_start = pd.Timestamp("2024-07-16 06:00")
zoom_end   = pd.Timestamp("2024-07-16 20:00")
zm         = (window.index >= zoom_start) & (window.index <= zoom_end)
daytime_window = y_window > 0

legend_handles = (
    [plt.Line2D([0],[0], color="black", linewidth=3, label="Actual")] +
    [plt.Line2D([0],[0], color=COLORS[n], linewidth=LINEWIDTHS[n],
                linestyle=LINESTYLES[n], label=n) for n in MODEL_NAMES]
)

# ── Fig 1: Full week timeline ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(24, 8))
ax.plot(window.index[daytime_window], y_window[daytime_window],
        color="black", linewidth=3.5, label="Actual", zorder=10)
for name in MODEL_NAMES:
    p = window_preds[name]
    ax.plot(window.index[daytime_window], p[daytime_window],
            color=COLORS[name], linewidth=LINEWIDTHS[name],
            linestyle=LINESTYLES[name], alpha=0.85, zorder=5)
ax.set_title("24-hr Forecast — All Models vs Actual  (Jul 15–22, 2024)", pad=14)
ax.set_ylabel("Power (kW)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%a %b %d"))
ax.xaxis.set_major_locator(mdates.DayLocator())
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")
ax.legend(handles=legend_handles, ncol=2, framealpha=0.5, loc="upper left")
plt.tight_layout()
plt.savefig("fig1_week_timeline.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → fig1_week_timeline.png")

# ── Fig 2: Zoom Jul 16 ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(24, 8))
ax.plot(window.index[zm], y_window[zm], color="black", linewidth=3.5, zorder=10, label="Actual")
for name in MODEL_NAMES:
    p = window_preds[name]
    ax.plot(window.index[zm], p[zm],
            color=COLORS[name], linewidth=LINEWIDTHS[name],
            linestyle=LINESTYLES[name], alpha=0.9, zorder=5)
ax.set_title("Zoom: Jul 16 (Problem Day) — Model Divergence", pad=14)
ax.set_ylabel("Power (kW)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
ax.xaxis.set_major_locator(mdates.HourLocator(byhour=[6,8,10,12,14,16,18,20]))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")
ax.legend(handles=legend_handles, ncol=2, framealpha=0.5, loc="upper left")
plt.tight_layout()
plt.savefig("fig2_zoom_jul16.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → fig2_zoom_jul16.png")

# ── Fig 3: Test RMSE% bar ─────────────────────────────────────────────────────
rmse_pct_vals = [results[n]["rmse_pct"] for n in MODEL_NAMES]
fig, ax = plt.subplots(figsize=(18, 8))
bars = ax.bar(MODEL_NAMES, rmse_pct_vals,
              color=[COLORS[n] for n in MODEL_NAMES], alpha=0.88, edgecolor="white", width=0.6)
ax.bar_label(bars, fmt="%.1f%%", fontsize=18, padding=5, fontweight="bold")
ax.set_title("Test RMSE%  (RMSE / mean daytime power × 100) — lower is better", pad=14)
ax.set_ylabel("RMSE%")
ax.set_ylim(0, max(rmse_pct_vals) * 1.22)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=25, ha="right")
plt.tight_layout()
plt.savefig("fig3_test_rmse.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → fig3_test_rmse.png")

# ── Fig 4: Val vs Test RMSE% ──────────────────────────────────────────────────
val_rmse_pct_vals = [results[n]["val_rmse_pct"] for n in MODEL_NAMES]
x = np.arange(len(MODEL_NAMES)); width = 0.38
fig, ax = plt.subplots(figsize=(20, 8))
bv = ax.bar(x - width/2, val_rmse_pct_vals, width,
            color="lightgrey", edgecolor="grey", hatch="///", linewidth=1.2)
bt = ax.bar(x + width/2, rmse_pct_vals, width,
            color=[COLORS[n] for n in MODEL_NAMES], alpha=0.9, edgecolor="white")
ax.bar_label(bv, fmt="%.1f%%", fontsize=15, padding=4)
ax.bar_label(bt, fmt="%.1f%%", fontsize=15, padding=4)
ax.set_xticks(x)
ax.set_xticklabels(MODEL_NAMES, rotation=25, ha="right")
ax.set_title("Validation RMSE% vs Test RMSE% — Generalisation Check", pad=14)
ax.set_ylabel("RMSE%")
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor="lightgrey", edgecolor="grey", hatch="///", label="Val (Jan–May 2024)"),
    Patch(facecolor="#888888",   edgecolor="white",             label="Test (Jun–Oct 2024)  [model colour]"),
], framealpha=0.5, loc="upper right")
plt.tight_layout()
plt.savefig("fig4_val_vs_test.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → fig4_val_vs_test.png")

# ── Fig 5: R² bar ─────────────────────────────────────────────────────────────
r2_vals = [results[n]["r2"] for n in MODEL_NAMES]
fig, ax = plt.subplots(figsize=(18, 8))
bars4 = ax.bar(MODEL_NAMES, r2_vals,
               color=[COLORS[n] for n in MODEL_NAMES], alpha=0.88, edgecolor="white", width=0.6)
ax.bar_label(bars4, fmt="%.3f", fontsize=18, padding=5, fontweight="bold")
ax.set_title("Test R²  (daytime only) — higher is better", pad=14)
ax.set_ylabel("R²")
ax.set_ylim(min(r2_vals) - 0.05, 1.1)
ax.axhline(0, color="black", linewidth=1, linestyle="--", alpha=0.4)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=25, ha="right")
plt.tight_layout()
plt.savefig("fig5_r2.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → fig5_r2.png")

# ── Fig 6: Per-day RMSE% heatmap ──────────────────────────────────────────────
hmap = np.array([daily_rmse_pct[n] for n in MODEL_NAMES], dtype=float)
fig, ax = plt.subplots(figsize=(18, 9))
im = ax.imshow(hmap, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(len(daily_df.index)))
ax.set_xticklabels(daily_df.index, rotation=30, ha="right")
ax.set_yticks(range(len(MODEL_NAMES)))
ax.set_yticklabels(MODEL_NAMES)
ax.set_title("Per-Day RMSE%  (Forecast Week Jul 15–21)", pad=14)
for i in range(len(MODEL_NAMES)):
    for j in range(len(daily_df.index)):
        val = hmap[i, j]
        txt = f"{val:.1f}%" if not np.isnan(val) else "—"
        ax.text(j, i, txt, ha="center", va="center", fontsize=16, fontweight="bold",
                color="black" if val < np.nanmax(hmap)*0.7 else "white")
cbar = plt.colorbar(im, ax=ax, fraction=0.025, pad=0.03)
cbar.set_label("RMSE%", fontsize=18)
cbar.ax.tick_params(labelsize=16)
plt.tight_layout()
plt.savefig("fig6_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → fig6_heatmap.png")

# ── Fig 7: 30-day forecast ────────────────────────────────────────────────────
best_ml = min([n for n in results if results[n]["model"] is not None and n not in BASELINE_NAMES],
              key=lambda n: results[n]["rmse_pct"])
best_bl = min(BASELINE_NAMES, key=lambda b: results[b]["rmse_pct"])

forecast_30_start = pd.Timestamp("2024-07-01")
forecast_30_end   = forecast_30_start + pd.Timedelta(days=30)
window_30   = df_test_feat[forecast_30_start:forecast_30_end]
X_window_30 = window_30[FEATURES]
y_window_30 = window_30[TARGET]

preds_30_raw = results[best_ml]["model"].predict(X_window_30)
preds_30     = nighttime_zero(preds_30_raw, window_30)
rmse_30      = rmse_pct(y_window_30.values, preds_30)

mask_30 = (df_test_feat.index >= forecast_30_start) & (df_test_feat.index <= forecast_30_end)
bl_30   = results[best_bl]["preds"][mask_30]

dt_30 = y_window_30 > 0
fig, ax = plt.subplots(figsize=(28, 9))
ax.plot(window_30.index[dt_30], y_window_30[dt_30],
        color="black", linewidth=2.5, label="Actual", zorder=10)
ax.plot(window_30.index[dt_30], preds_30[dt_30],
        color=COLORS[best_ml], linewidth=2.2, linestyle=LINESTYLES[best_ml],
        alpha=0.85, label=f"{best_ml}  RMSE%={rmse_30:.1f}%", zorder=5)
ax.plot(window_30.index[dt_30], bl_30[dt_30],
        color=COLORS[best_bl], linewidth=1.8, linestyle=LINESTYLES[best_bl],
        alpha=0.75, label=f"{best_bl} baseline", zorder=4)
ax.set_title(f"30-Day Forecast — {best_ml} vs {best_bl} baseline  (Jul 2024)", pad=14)
ax.set_ylabel("Power (kW)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=3))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")
ax.legend(framealpha=0.5, loc="upper right")
plt.tight_layout()
plt.savefig("fig7_30day_forecast.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → fig7_30day_forecast.png")


## 9. Additional Diagnostics

In [ ]:
import matplotlib as mpl
mpl.rcParams.update({
    "font.size":         18,
    "axes.titlesize":    22,
    "axes.labelsize":    20,
    "xtick.labelsize":   17,
    "ytick.labelsize":   17,
    "legend.fontsize":   16,
    "axes.linewidth":    1.4,
    "axes.spines.top":   False,
    "axes.spines.right": False,
})

COLORS = {
    "LightGBM":          "#2196F3",
    "XGBoost":           "#FF5722",
    "RandomForest":      "#4CAF50",
    "Ridge":             "#9C27B0",
    "LinearReg":         "#00BCD4",  # cyan
    "MLP":               "#FF9800",   # only shown if USE_MLP=True
    "Persistence":       "#607D8B",
    "Climatology":       "#795548",
    "SmartPersistence":  "#009688",
    "Stacking":          "#F44336",  # red — the ensemble
}
LINESTYLES = {
    "LightGBM":         (0, ()),
    "XGBoost":          (0, (6, 2)),
    "RandomForest":     (0, (2, 2)),
    "Ridge":            (0, (4, 2, 1, 2)),
    "LinearReg":        (0, (2, 1)),
    "MLP":              (0, (1, 1)),
    "Persistence":      (0, (3, 1, 1, 1, 1, 1)),
    "Climatology":      (0, (5, 5)),
    "SmartPersistence": (0, (3, 1)),
    "Stacking":         (0,()),      # solid red — ensemble
}
LINEWIDTHS = {
    "LightGBM": 2.5, "XGBoost": 2.2, "RandomForest": 2.2, "Ridge": 2.2, "LinearReg": 2.0,
    "MLP": 2.2, "Persistence": 1.8, "Climatology": 1.8, "SmartPersistence": 1.8,
    "Stacking":          3.0,
}
# Only show models that were actually trained
MODEL_NAMES = [n for n in COLORS if n in results]


MODEL_NAMES = list(results.keys())

# ── Fig 8: Residual distribution ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(20, 8))
for name in MODEL_NAMES:
    dt = y_test > 10
    residuals = y_test.values[dt] - results[name]["preds"][dt]
    ax.hist(residuals, bins=80, alpha=0.35, color=COLORS[name],
            label=name, density=True, linewidth=0)
ax.axvline(0, color="black", linewidth=2, linestyle="--")
ax.set_title("Residual Distribution — Daytime Test Set", pad=14)
ax.set_xlabel("Residual (kW)")
ax.set_ylabel("Density")
ax.legend(ncol=2, framealpha=0.4)
plt.tight_layout()
plt.savefig("fig8_residuals.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → fig8_residuals.png")

# ── Fig 9: Monthly RMSE% ──────────────────────────────────────────────────────
months = df_test_feat.index.to_period("M").unique()
x = np.arange(len(months))
width = 0.1
fig, ax = plt.subplots(figsize=(20, 8))
for i, name in enumerate(MODEL_NAMES):
    monthly_errs = []
    for m in months:
        mask = df_test_feat.index.to_period("M") == m
        yt = y_test[mask].values
        yp = results[name]["preds"][mask]
        monthly_errs.append(rmse_pct(yt, yp))
    ax.bar(x + i * width, monthly_errs, width,
           color=COLORS[name], alpha=0.88, label=name, edgecolor="white")
ax.set_xticks(x + width * len(MODEL_NAMES) / 2)
ax.set_xticklabels([str(m) for m in months], rotation=25, ha="right")
# Mark anomaly days on the monthly plot
anomaly_months = {d.strftime('%Y-%m') for d in anomaly_dates_test}
ax.set_title("Monthly RMSE% — Test Set (Jun–Oct 2024)", pad=14)
ax.set_ylabel("RMSE%")
ax.legend(ncol=2, framealpha=0.4)
plt.tight_layout()
plt.savefig("fig9_monthly_rmse.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → fig9_monthly_rmse.png")

# ── Fig 10: Mean hourly residual profile ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(20, 8))
for name in MODEL_NAMES:
    err_df = pd.DataFrame({
        "hour":     df_test_feat.index.hour,
        "residual": y_test.values - results[name]["preds"],
    })
    profile = err_df.groupby("hour")["residual"].mean()
    ax.plot(profile.index, profile.values,
            color=COLORS[name], linewidth=LINEWIDTHS[name],
            linestyle=LINESTYLES[name], label=name)
ax.axhline(0, color="black", linewidth=1.5, linestyle="--", alpha=0.5)
ax.set_title("Mean Hourly Residual Profile — Test Set", pad=14)
ax.set_xlabel("Hour of day")
ax.set_ylabel("Mean residual (kW)")
ax.set_xticks(range(0, 24, 2))
ax.legend(ncol=2, framealpha=0.4)
plt.tight_layout()
plt.savefig("fig10_hourly_residual.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → fig10_hourly_residual.png")

# ── Fig 11: Rolling CV RMSE% boxplot ──────────────────────────────────────────
cv_model_names = [n for n in MODEL_NAMES if n in cv_dfs]
bp_data = [cv_dfs[n]["rmse_pct"].values for n in cv_model_names]
fig, ax = plt.subplots(figsize=(18, 8))
bp = ax.boxplot(bp_data, patch_artist=True, notch=False,
                medianprops=dict(color="black", linewidth=2.5),
                whiskerprops=dict(linewidth=1.8),
                capprops=dict(linewidth=1.8),
                flierprops=dict(marker="o", markersize=5, alpha=0.4))
for patch, name in zip(bp["boxes"], cv_model_names):
    patch.set_facecolor(COLORS[name])
    patch.set_alpha(0.8)
ax.set_xticklabels(cv_model_names, rotation=20, ha="right")
ax.set_title("Rolling CV RMSE% Distribution — Training Pool", pad=14)
ax.set_ylabel("RMSE%")
plt.tight_layout()
plt.savefig("fig11_cv_boxplot.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → fig11_cv_boxplot.png")


## 10. Feature Importance (Tree Models)

In [ ]:
tree_models = [n for n in ["LightGBM","XGBoost","RandomForest"] if n in results]

import matplotlib as mpl
mpl.rcParams.update({"font.size": 16, "axes.titlesize": 20,
                     "axes.labelsize": 17, "xtick.labelsize": 14, "ytick.labelsize": 14})

from matplotlib.patches import Patch
legend_els = [
    Patch(facecolor="#2196F3", label="Lag / power rolling"),
    Patch(facecolor="#FF9800", label="Cyclical time"),
    Patch(facecolor="#9C27B0", label="Derived (irradiance ratios, wind)"),
    Patch(facecolor="#4CAF50", label="Raw NWP weather"),
]

for name in tree_models:
    model = results[name]["model"]
    importances = model.feature_importances_
    imp_df = (pd.DataFrame({"feature": FEATURES, "importance": importances})
                .sort_values("importance"))

    def feat_color(f):
        if "lag_" in f or "pow_roll" in f:                                    return "#2196F3"
        if any(x in f for x in ["sin_","cos_","hour","month"]):               return "#FF9800"
        if any(x in f for x in ["roll","wind_u","wind_v","ratio","frac"]):    return "#9C27B0"
        return "#4CAF50"

    colors_f = [feat_color(f) for f in imp_df["feature"]]
    fig, ax = plt.subplots(figsize=(18, 14))
    ax.barh(imp_df["feature"], imp_df["importance"], color=colors_f, alpha=0.88, edgecolor="white")
    ax.set_title(f"Feature Importance — {name}", pad=14)
    ax.set_xlabel("Importance")
    ax.legend(handles=legend_els, fontsize=14, loc="lower right")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    fname = f"fig12_feat_imp_{name.lower()}.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {fname}")


## 11. Summary

In [ ]:
# ── Mean daily RMSE ──────────────────────────────────────────────────────────
print()
print("  MEAN DAILY RMSE (kW) — average per-day RMSE across test period:")
print(f"  {'Model':<22} {'Mean Daily RMSE (kW)':>22} {'Std Dev':>12}")
print("  " + "-"*60)

# Define model name lists locally — avoids dependency on prior cells
_ml_names_local = [n for n in results
                   if results[n]["model"] is not None
                   and n not in BASELINE_NAMES
                   and n != "Stacking"]

all_model_names = (list(_ml_names_local) +
                   (["Stacking"] if "Stacking" in results else []) +
                   list(BASELINE_NAMES))

for name in all_model_names:
    if name not in results:
        continue
    preds = results[name]["preds"]
    daily_rmses = []
    for d in pd.date_range(df_test_feat.index.min().date(),
                             df_test_feat.index.max().date(), freq="D"):
        mask = df_test_feat.index.date == d.date()
        y_d  = y_test.values[mask]
        p_d  = preds[mask]
        dt   = y_d > 0          # daytime only
        if dt.sum() == 0:
            continue
        daily_rmses.append(np.sqrt(np.mean((p_d[dt] - y_d[dt]) ** 2)))
    mean_dr = np.mean(daily_rmses)
    std_dr  = np.std(daily_rmses)
    print(f"  {name:<22} {mean_dr:>20.2f} kW  {std_dr:>10.2f} kW")

print()
print("  Note: Mean Daily RMSE = average of each day's individual RMSE(kW),")
print("        computed on daytime rows only. Std Dev shows day-to-day variability.")
print("=" * 75)
print("  FINAL MODEL COMPARISON SUMMARY")
print(f"  USE_MLP = {USE_MLP}")
print("  Train: 2022-03-23 → 2023-11-09 | Val: 2024-H1 | Test: 2024-H2")
print("=" * 75)
print(summary.round(3).to_string())
print()

ml_names = [n for n in results if results[n]["model"] is not None
            and n not in BASELINE_NAMES and n != "Stacking"]

# ── Full comparison table: all models, all metrics ───────────────────────────
all_names = (list(_ml_names_local) +
             (["Stacking"] if "Stacking" in results else []) +
             list(BASELINE_NAMES))

# Compute mean daytime actual — must match threshold used in rmse_pct (> 0)
# Note: sanity check earlier printed 357.1 kW using > 10 kW threshold.
# rmse_pct() uses > 0, giving a slightly lower mean (~333 kW) because
# it includes small dawn/dusk values. Both thresholds are used here.
dt_mask          = y_test.values > 0    # matches rmse_pct() — used for RMSE%
dt_mask_strict   = y_test.values > 10   # strict — excludes dawn/dusk transitions
mean_day_actual  = y_test.values[dt_mask].mean()
mean_day_strict  = y_test.values[dt_mask_strict].mean()

print("  FULL MODEL COMPARISON — All Models, All Metrics:")
print(f"  {'Model':<22} {'RMSE%':>8} {'RMSE kW':>12} {'RMSE kW':>12} {'MWOL':>10} {'FE%':>8}")
print(f"  {'':22} {'':>8} {'(daytime)':>12} {'(all hrs)':>12} {'($)':>10} {'':>8}")
print("  " + "─"*76)
for name in all_names:
    if name not in results:
        continue
    r    = results[name]
    rp   = r.get("rmse_pct",    float("nan"))
    rk   = r.get("rmse_kw",     float("nan"))
    rka  = r.get("rmse_kw_all", float("nan"))
    tag  = " ◄ ML" if (r["model"] is not None or name == "Stacking") else " ◄ baseline"
    # Get MWOL from financial_results if available
    try:
        mwol = financial_results[name]["mwol"]
        fe   = financial_results[name]["fe_pct"]
        print(f"  {name:<22} {rp:>7.1f}%  {rk:>10.1f}kW  {rka:>10.1f}kW  "
              f"${mwol:>8,.0f}  {fe:>7.2f}%{tag}")
    except (KeyError, NameError):
        print(f"  {name:<22} {rp:>7.1f}%  {rk:>10.1f}kW  {rka:>10.1f}kW{tag}")
print("  " + "─"*76)
print()

# ── Worked example: how RMSE% is computed ────────────────────────────────────
best_name = min(_ml_names_local, key=lambda n: results[n].get("rmse_kw", float("inf")))
r_ex      = results[best_name]
rk_ex     = r_ex.get("rmse_kw", float("nan"))
rp_ex     = r_ex.get("rmse_pct", float("nan"))

print("  HOW RMSE% IS COMPUTED — worked example using best model:")
print(f"  Model: {best_name}")
print()
print("  Step 1 — compute residuals for every daytime 15-min interval:")
print("           residual_i = predicted_i − actual_i   (where actual_i > 0)")
print()
print("  Step 2 — square each residual and take the mean:")
print("           MSE = mean( residual_i² )")
print()
print("  Step 3 — take the square root → RMSE in kW:")
print(f"           RMSE = sqrt(MSE) = {rk_ex:.2f} kW")
print()
print("  Step 4 — divide by the mean daytime actual power:")
print(f"           mean daytime actual = {mean_day_actual:.2f} kW")
print(f"           RMSE% = {rk_ex:.2f} / {mean_day_actual:.2f} × 100 = {rp_ex:.1f}%")
print()
print("  Interpretation:")
print(f"           On average, the forecast is off by ±{rk_ex:.1f} kW per 15-min slot")
print(f"           during the day. That is {rp_ex:.1f}% of the system's mean daytime output")
print(f"           ({mean_day_actual:.1f} kW). The system capacity is {SYSTEM_CAPACITY_KW:.0f} kW,")
print(f"           so the error is {rk_ex/SYSTEM_CAPACITY_KW*100:.1f}% of installed capacity.")
print()

best_test = summary.loc[[n for n in summary.index if n in _ml_names_local or n == "Stacking"],
                         "Test RMSE% (all days)"].idxmin()
best_bl   = min(BASELINE_NAMES, key=lambda b: results[b]["rmse_pct"])
best_bl_rmse = results[best_bl]["rmse_pct"]

print(f"  Best ML Test RMSE%  : {best_test} ({summary.loc[best_test,'Test RMSE% (all days)']:.2f}%)")
print(f"  Best ML RMSE (day)  : {best_test} ({results[best_test].get('rmse_kw',float('nan')):.2f} kW)")
print(f"  Best ML RMSE (all)  : {best_test} ({results[best_test].get('rmse_kw_all',float('nan')):.2f} kW)")
print(f"  Best baseline       : {best_bl} ({best_bl_rmse:.2f}%  {results[best_bl].get('rmse_kw',float('nan')):.2f} kW daytime)")
print()
print("  Forecast Skill Score (FSS = 1 - model_RMSE% / baseline_RMSE%):")
for name in list(_ml_names_local) + (["Stacking"] if "Stacking" in results else []):
    fss = 1 - results[name]["rmse_pct"] / best_bl_rmse
    print(f"    {name:<15}  FSS={fss:+.3f}  "
          f"RMSE={results[name].get('rmse_kw',float('nan')):.1f}kW(day)  "
          f"{results[name].get('rmse_kw_all',float('nan')):.1f}kW(all)")
print("=" * 75)

# ── Average daily energy production ──────────────────────────────────────────
# Energy per 15-min interval (kWh) = power (kW) × 0.25 h
# Sum per day = daily energy (kWh); divide by 1000 for MWh

daily_energy_kwh = (
    pd.Series(y_test.values, index=df_test_feat.index)
      .resample("D").sum() * 0.25          # kW → kWh
)

print()
print("  AVERAGE DAILY ENERGY PRODUCTION — Test Set (Jun–Oct 2024):")
print(f"  Mean daily energy  : {daily_energy_kwh.mean():>8.1f} kWh  "
      f"/ {daily_energy_kwh.mean()/1000:.3f} MWh")
print(f"  Median daily energy: {daily_energy_kwh.median():>8.1f} kWh  "
      f"/ {daily_energy_kwh.median()/1000:.3f} MWh")
print(f"  Std dev            : {daily_energy_kwh.std():>8.1f} kWh")
print(f"  Min day            : {daily_energy_kwh.min():>8.1f} kWh  "
      f"({daily_energy_kwh.idxmin().date()})")
print(f"  Max day            : {daily_energy_kwh.max():>8.1f} kWh  "
      f"({daily_energy_kwh.idxmax().date()})")
print(f"  Total test period  : {daily_energy_kwh.sum():>8.1f} kWh  "
      f"/ {daily_energy_kwh.sum()/1000:.2f} MWh")
print(f"  Days in test set   : {len(daily_energy_kwh)}")
print()
print("  Capacity factor (mean daily energy / theoretical max):")
theoretical_max_kwh = SYSTEM_CAPACITY_KW * 24   # kW × 24h
cf = daily_energy_kwh.mean() / theoretical_max_kwh * 100
print(f"  Theoretical max    : {theoretical_max_kwh:>8.1f} kWh/day  "
      f"(system at full capacity 24h)")
print(f"  Capacity factor    : {cf:>7.1f}%")
print("=" * 75)


## Next Steps
- Run `03_financial_analysis.ipynb` for MWOL, bidding strategy and financial plots
- The `results` dict from this notebook is needed — run cells in order
